Dataset: https://www.kaggle.com/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024

Covers: `01_overview.py` charts

## Libraries

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## Paths

In [ ]:
KAGGLE_DIR = "/kaggle/input/1-3m-linkedin-jobs-and-skills-2024"
LOCAL_DIR = "."

data_dir = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else LOCAL_DIR

postings_path = f"{data_dir}/linkedin_job_postings.csv"
skills_path = f"{data_dir}/job_skills.csv"
summary_path = f"{data_dir}/job_summary.csv"

PARQUET_PATH = "/tmp/data/merged.parquet"
SAMPLE_N = 200_000

print("data_dir:", data_dir)

## Shared Contracts — processor.py

In [ ]:
def load_postings(path: str, sample_n: int) -> pd.DataFrame:
    df = pd.read_csv(path, nrows=sample_n)
    df["job_title"] = df["job_title"].astype(str).str.lower().str.strip()
    df["company"] = df["company"].fillna("Unknown Company")
    df["job_location"] = df["job_location"].fillna("Unknown Location")
    df = df.drop(columns=["got_summary", "got_ner", "is_being_worked"], errors="ignore")
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "job_link"])
    processed = pd.to_datetime(
        df["last_processed_time"], dayfirst=True, format="mixed", utc=True
    )
    df["date"] = processed.dt.date
    df["hour"] = processed.dt.hour
    df["day"] = processed.dt.day
    df["day_of_week"] = processed.dt.day_name()
    df = df.drop(columns=["last_processed_time"])
    return df


def aggregate_skills(path: str) -> pd.DataFrame:
    skills_raw = pd.read_csv(path)
    return (
        skills_raw.groupby("job_link")["job_skills"]
        .apply(list)
        .reset_index()
        .rename(columns={"job_skills": "skills_list"})
    )


def load_summary(path: str) -> pd.DataFrame:
    return pd.read_csv(path, usecols=["job_link", "job_summary"])


def merge_datasets(
    postings: pd.DataFrame,
    skills_agg: pd.DataFrame,
    summary: pd.DataFrame,
) -> pd.DataFrame:
    df = postings.merge(skills_agg, on="job_link", how="left")
    df = df.merge(summary, on="job_link", how="left")
    df["skills_list"] = df["skills_list"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df["job_summary"] = df["job_summary"].fillna("")
    return df


def get_merged(parquet_path: str, sample_n: int) -> pd.DataFrame:
    if os.path.exists(parquet_path):
        return pd.read_parquet(parquet_path)
    postings = load_postings(postings_path, sample_n)
    skills_agg = aggregate_skills(skills_path)
    summary = load_summary(summary_path)
    df = merge_datasets(postings, skills_agg, summary)
    os.makedirs(os.path.dirname(parquet_path), exist_ok=True)
    df.to_parquet(parquet_path, index=False)
    return df

## Load Data

In [ ]:
df = get_merged(PARQUET_PATH, SAMPLE_N)

print("shape:", df.shape)
print("columns:", df.columns.tolist())
print(df.isnull().sum()[lambda s: s > 0])

## Visualization Theme

In [ ]:
bright_palette = [
    "#4CC9F0",
    "#4895EF",
    "#4361EE",
    "#7B2CBF",
    "#9D4EDD",
    "#C77DFF",
    "#F72585",
    "#480CA8",
]

sns.set_theme(style="whitegrid", context="talk", palette=bright_palette)
plt.rcParams["figure.facecolor"] = "#F8FAFF"
plt.rcParams["axes.facecolor"] = "#FFFFFF"
plt.rcParams["axes.edgecolor"] = "#D6D6E5"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["grid.linestyle"] = "--"

## Top 10 Job Titles

In [ ]:
top_jobs = df["job_title"].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_jobs.values, y=top_jobs.index, palette=bright_palette)
plt.title("Top 10 Job Titles")
plt.xlabel("Number of Listings")
plt.ylabel("Job Title")
plt.tight_layout()
plt.show()

## Job Processing Time by Day of Week

In [ ]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
processing_time = df["day_of_week"].value_counts().reindex(day_order).dropna()

plt.figure(figsize=(10, 6))
sns.barplot(x=processing_time.index, y=processing_time.values, palette=bright_palette)
plt.title("Job Processing Time by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Number of Postings")
plt.tight_layout()
plt.show()

## Top 10 Companies by Job Count

In [ ]:
top_companies = df["company"].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_companies.values, y=top_companies.index, palette=bright_palette)
plt.title("Top 10 Companies by Job Postings")
plt.xlabel("Number of Postings")
plt.ylabel("Company")
plt.tight_layout()
plt.show()

## Job Level Distribution

In [ ]:
job_levels = df["job_level"].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=job_levels.values, y=job_levels.index, palette=bright_palette)
plt.title("Job Level Distribution")
plt.xlabel("Number of Postings")
plt.ylabel("Job Level")
plt.tight_layout()
plt.show()

## Top 10 Job Locations

In [ ]:
top_locations = df["job_location"].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_locations.values, y=top_locations.index, palette=bright_palette)
plt.title("Top 10 Job Locations")
plt.xlabel("Number of Postings")
plt.ylabel("Location")
plt.tight_layout()
plt.show()

## Job Postings by Hour of Day

In [ ]:
hourly = df["hour"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
sns.barplot(x=hourly.index, y=hourly.values, color="#4CC9F0")
plt.title("Job Postings by Hour of Day")
plt.xlabel("Hour (UTC)")
plt.ylabel("Number of Postings")
plt.tight_layout()
plt.show()

## Job Openings by Day of Month

In [ ]:
plt.figure(figsize=(12, 6))
sns.violinplot(x=df["day"].head(10000), color="#4CC9F0")
plt.title("Job Openings by Day of Month")
plt.xlabel("Day of Month")
plt.tight_layout()
plt.show()

## Search Position Distribution

In [ ]:
search_counts = df.groupby("search_position").size().head(20)

plt.figure(figsize=(12, 6))
sns.histplot(search_counts, kde=True, color="#7B2CBF", edgecolor="k")
plt.title("Distribution of Job Count Across Search Positions")
plt.xlabel("Number of Postings")
plt.tight_layout()
plt.show()